# 02 · Compare the right targets / 比较正确的任务目标

**Goal / 目标**: inspect E02, E03 and E07 without retraining or market-data redistribution. / 在不重新训练或分发市场原始数据的情况下检查 E02、E03 和 E07。

Read [case 02](../docs/cases/02-representations.md) / [中文案例](../docs/cases/02-representations_zh.md). Values are selected archived summaries; this notebook demonstrates reading and arithmetic. / 数值来自精选历史摘要，本文件演示读取与运算。

In [ ]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'evidence/E01.json').is_file())
def read(name):
    return json.loads((ROOT / 'evidence' / name).read_text(encoding='utf-8'))


In [ ]:
e02, e03, e07 = read('E02.json'), read('E03.json'), read('E07.json')
for e in (e02, e03, e07):
    print(e['experiment_id'], e['task_id'], e['statistical_unit'])
for comparison in e02['comparison_family']:
    print('E02', comparison['comparison'], comparison['effect'], comparison['ci95'])


E02 estimates five-session downside risk; E03 predicts the stock-return task. Their errors have different meanings. Shared quantum vocabulary does not make targets interchangeable. / E02 估计五日下行风险，E03 对应股票收益预测，二者误差含义不同。

For E03, lower MAE is favorable. / E03 的 MAE 越小越好。

In [ ]:
endpoints = e03['endpoints']
selected = endpoints['selected_directional']['mae']['mean']
classical = endpoints['classical_validation_graph']['mae']['mean']
fixed = endpoints['fixed_directional']['mae']['mean']
print({'selected_mae': selected, 'classical_mae': classical, 'difference': selected-classical})
assert selected > classical and selected == fixed


The selected directional method has higher MAE than the selected classical reference. Selection equals the fixed directional map here. / 本任务中选择的方向量子方法 MAE 高于经典参照，且结果与固定方向映射一致。

E07 tests the financial endpoint with a specified execution proxy and cost. / E07 在规定执行代理与成本下考察金融端点。

In [ ]:
portfolio = {e['method']: e for e in e07['endpoints'] if e['cost'] == .001}
for method, e in portfolio.items():
    print(method, e['mean_seed_cvar95'])
difference = portfolio['full']['mean_seed_cvar95'] - portfolio['classical']['mean_seed_cvar95']
assert abs(difference - e07['primary_comparisons'][0]['effect']) < 1e-12
assert difference > 0
print('CVaR95 difference:', difference)


## Exercise / 练习

Write one claim for each experiment containing its target, baseline, sample unit and limitation. Explain why an improvement on E02 does not establish improved portfolio tail risk. / 为每个实验写一句包含目标、基线、样本单位与边界的结论，并解释 E02 改善为什么不能推出组合尾部风险改善。

Confidence intervals are historical outputs. Reconstructing them needs the dated observations and bootstrap configuration. / 区间是历史输出，重建需要按日观测和 bootstrap 配置。